# Medical Data With ifcfill And SeqTrees

This notebook demonstrates a fuller workflow:

1. Load public Synthea healthcare sample data from a URL.
2. Keep a patient-level table with dates, categorical values, numeric values, and missing values.
3. Use `ifcfill` to impute missing values, label-encode categorical values, and convert datetimes into numeric IFC columns.
4. Fit SeqTrees, sample the same number of rows, and pass the result through `ifcfill.inverse_transform()`.

The Synthea sample data is generated healthcare data, not real patient PHI. That makes it suitable for a public tutorial while still looking like a medical table.

In [10]:
# Run the following in a fresh notebook environment.
# %pip install -q "seqtrees[examples]"


In [11]:
from pathlib import Path
from zipfile import ZipFile
import sys

import pandas as pd
from IPython.display import display


# When running this notebook from the repository checkout, prefer the local SeqTrees source tree.
repo_root = Path.cwd()
if (repo_root / "src" / "seqtrees").exists():
    sys.path.insert(0, str(repo_root / "src"))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)

from ifcfill import IFCTransformer
from seqtrees import SequentialTreeSynthesizer

In [12]:
DATA_URL = "https://github.com/synthetichealth/synthea-sample-data/raw/main/downloads/synthea_sample_data_csv_apr2020.zip"
DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "synthea_sample_data_csv_apr2020.zip"

DATA_DIR.mkdir(exist_ok=True)
if not ZIP_PATH.exists():
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

with ZipFile(ZIP_PATH) as zf:
    csv_names = zf.namelist()
    patients_name = next(name for name in csv_names if name.endswith("patients.csv"))
    with zf.open(patients_name) as f:
        raw_patients = pd.read_csv(f)

print(f"Raw patients: {raw_patients.shape[0]:,} rows x {raw_patients.shape[1]:,} columns")
display(raw_patients.head())

Raw patients: 1,171 rows x 25 columns


,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE
0,1d604da9-9a81-4ba9-80c2-de3375d59b40,1989-05-25,NaN,999-76-6866,S99984236,X19277260X,Mr.,José Eduardo181,Gómez206,NaN,NaN,M,white,hispanic,M,Marigot Saint Andrew Parish DM,427 Balistreri Way Unit 19,Chicopee,Massachusetts,Hampden County,1013.0,42.228354,-72.562951,271227.08,1334.88
1,034e9e3b-2def-4559-bb2a-7850888ae060,1983-11-14,NaN,999-73-5361,S99962402,X88275464X,Mr.,Milo271,Feil794,NaN,NaN,M,white,nonhispanic,M,Danvers Massachusetts US,422 Farrell Path Unit 69,Somerville,Massachusetts,Middlesex County,2143.0,42.360697,-71.126531,793946.01,3204.49
2,10339b10-3cd1-4ac3-ac13-ec26728cb592,1992-06-02,NaN,999-27-3385,S99972682,X73754411X,Mr.,Jayson808,Fadel536,NaN,NaN,M,white,nonhispanic,M,Springfield Massachusetts US,1056 Harris Lane Suite 70,Chicopee,Massachusetts,Hampden County,1020.0,42.181642,-72.608842,574111.90,2606.40
3,8d4c4326-e9de-4f45-9a4c-f8c36bff89ae,1978-05-27,NaN,999-85-4926,S99974448,X40915583X,Mrs.,Mariana775,Rutherford999,NaN,Williamson769,M,white,nonhispanic,F,Yarmouth Massachusetts US,999 Kuhn Forge,Lowell,Massachusetts,Middlesex County,1851.0,42.636143,-71.343255,935630.30,8756.19
4,f5dcd418-09fe-4a2f-baa0-3da800bd8c3a,1996-10-18,NaN,999-60-7372,S99915787,X86772962X,Mr.,Gregorio366,Auer97,NaN,NaN,NaN,white,nonhispanic,M,Patras Achaea GR,1050 Lindgren Extension Apt 38,Boston,Massachusetts,Suffolk County,2135.0,42.352434,-71.028610,598763.07,3772.20


In [13]:
# A compact but mixed-type patient table: dates, strings, floats, and missing DEATHDATE values.
columns = [
    "BIRTHDATE",
    "DEATHDATE",
    "MARITAL",
    "RACE",
    "ETHNICITY",
    "GENDER",
    "BIRTHPLACE",
    "CITY",
    "STATE",
    "HEALTHCARE_EXPENSES",
    "HEALTHCARE_COVERAGE",
]

patients = raw_patients[columns].copy()
patients["BIRTHDATE"] = pd.to_datetime(patients["BIRTHDATE"], errors="coerce")
patients["DEATHDATE"] = pd.to_datetime(patients["DEATHDATE"], errors="coerce")

# Keep the example quick for notebooks while preserving enough rows for synthesis.
patients = patients.sample(min(len(patients), 1000), random_state=42).reset_index(drop=True)

display(patients.head())

missing_counts = patients.isna().sum().sort_values(ascending=False).head(8)
display(missing_counts.rename("missing_count").to_frame())

,BIRTHDATE,DEATHDATE,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,CITY,STATE,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE
0,1958-05-20,NaT,M,white,nonhispanic,M,Gloucester Massachusetts US,Newton,Massachusetts,1295801.20,6263.48
1,1961-11-12,NaT,S,white,nonhispanic,F,Boston Massachusetts US,Winchester,Massachusetts,1287395.16,8873.55
2,2017-08-18,NaT,NaN,asian,nonhispanic,F,Norton Massachusetts US,Maynard,Massachusetts,61200.00,1291.60
3,2015-01-03,NaT,NaN,white,nonhispanic,M,Nantucket Massachusetts US,Westford,Massachusetts,110983.92,1857.40
4,1979-03-26,NaT,M,asian,nonhispanic,F,Beijing Beijing Municipality CN,Weymouth,Massachusetts,38180.48,1383.20


,missing_count
DEATHDATE,855
MARITAL,321
BIRTHDATE,0
RACE,0
ETHNICITY,0
GENDER,0
BIRTHPLACE,0
CITY,0


`IFCTransformer` prepares the mixed medical table for a numeric synthesizer. It fills missing values using `ifcfill`'s default strategies, converts date columns into day offsets from the chosen anchor date, label-encodes categorical columns with `cat_encoding="label"`, and remembers all metadata needed to restore dates and category labels later with `inverse_transform()`.

In [14]:
ifc = IFCTransformer(
    col_types={
        "BIRTHDATE": "datetime",
        "DEATHDATE": "datetime",
        "MARITAL": "categorical",
        "RACE": "categorical",
        "ETHNICITY": "categorical",
        "GENDER": "categorical",
        "BIRTHPLACE": "categorical",
        "CITY": "categorical",
        "STATE": "categorical",
        "HEALTHCARE_EXPENSES": "float",
        "HEALTHCARE_COVERAGE": "float",
    },
    cat_encoding="label",
    datetime_anchor="1900-01-01",
    datetime_unit="D",
)

ifc_table = ifc.fit_transform(patients)

display(ifc.missing_report_.sort_values("missing_fraction", ascending=False).head(10))
display(ifc_table.head())

,column,type,missing_count,missing_fraction
1,DEATHDATE,datetime,855,0.855
2,MARITAL,categorical,321,0.321
0,BIRTHDATE,datetime,0,0.000
3,RACE,categorical,0,0.000
4,ETHNICITY,categorical,0,0.000
5,GENDER,categorical,0,0.000
6,BIRTHPLACE,categorical,0,0.000
7,CITY,categorical,0,0.000
8,STATE,constant,0,0.000
9,HEALTHCARE_EXPENSES,float,0,0.000


,BIRTHDATE,DEATHDATE,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,CITY,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE
0,21323,37492,0,5,2,1,82,135,1295801.20,6263.48
1,22595,37492,1,5,2,0,29,219,1287395.16,8873.55
2,42963,37492,2,1,2,0,168,115,61200.00,1291.60
3,42005,37492,2,5,2,1,154,209,110983.92,1857.40
4,28938,37492,0,1,2,0,20,213,38180.48,1383.20


In [15]:
ifc_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   BIRTHDATE            1000 non-null   int64  
 1   DEATHDATE            1000 non-null   int64  
 2   MARITAL              1000 non-null   int64  
 3   RACE                 1000 non-null   int64  
 4   ETHNICITY            1000 non-null   int64  
 5   GENDER               1000 non-null   int64  
 6   BIRTHPLACE           1000 non-null   int64  
 7   CITY                 1000 non-null   int64  
 8   HEALTHCARE_EXPENSES  1000 non-null   float64
 9   HEALTHCARE_COVERAGE  1000 non-null   float64
dtypes: float64(2), int64(8)
memory usage: 78.3 KB


`ifcfill` has already done the missing-value work, label-encoded categorical columns, and converted dates to numeric offsets. The result is an IFC table containing only integer and float values, so SeqTrees can infer the variable types directly.

In [16]:
model = SequentialTreeSynthesizer(
    optimize_order=True,
    tree_backend="auto",
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)

model.fit(ifc_table)
synthetic_ifc = model.sample(len(ifc_table), random_state=101, as_dataframe=True, n_jobs=-1)

print("Variable order:", model.get_variable_order())
display(synthetic_ifc.head())

Variable order: ['ETHNICITY', 'RACE', 'GENDER', 'DEATHDATE', 'MARITAL', 'CITY', 'BIRTHPLACE', 'BIRTHDATE', 'HEALTHCARE_COVERAGE', 'HEALTHCARE_EXPENSES']


,BIRTHDATE,DEATHDATE,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,CITY,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE
0,26679,34419,0,5,2,0,8,147,1054359.51,12398.31
1,19878,37492,1,5,2,1,185,133,183031.24,354.96
2,42731,37492,2,5,2,0,133,38,8747.59,1033.28
3,31804,37492,2,5,2,0,91,41,830702.61,5769.22
4,23228,37492,0,1,2,1,120,85,625541.72,6263.48


In [17]:
synthetic_patients = ifc.inverse_transform(
    synthetic_ifc,
    restore_missing=True,
    random_state=202,
)

print(f"Synthetic patients: {synthetic_patients.shape[0]:,} rows x {synthetic_patients.shape[1]:,} columns")
display(synthetic_patients.head())

Synthetic patients: 1,000 rows x 11 columns


,BIRTHDATE,DEATHDATE,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,CITY,STATE,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE
0,1973-01-17,NaT,M,white,nonhispanic,F,Arlington Massachusetts US,Norwood,Massachusetts,1054359.51,12398.31
1,1954-06-05,NaT,S,white,nonhispanic,M,Plymouth Massachusetts US,New Bedford,Massachusetts,183031.24,354.96
2,2016-12-29,NaT,NaN,white,nonhispanic,F,Marshfield Massachusetts US,Chelmsford,Massachusetts,8747.59,1033.28
3,1987-01-29,NaT,NaN,white,nonhispanic,F,Hamilton Massachusetts US,Chicopee,Massachusetts,830702.61,5769.22
4,1963-08-07,NaT,M,asian,nonhispanic,M,Lowell Massachusetts US,Holden,Massachusetts,625541.72,6263.48


In [18]:
summary = pd.DataFrame(
    {
        "original_missing": patients.isna().mean(),
        "synthetic_missing": synthetic_patients.isna().mean(),
    }
)

display(summary.sort_values("original_missing", ascending=False))

,original_missing,synthetic_missing
DEATHDATE,0.855,0.855
MARITAL,0.321,0.339
BIRTHDATE,0.000,0.000
RACE,0.000,0.000
ETHNICITY,0.000,0.000
GENDER,0.000,0.000
BIRTHPLACE,0.000,0.000
CITY,0.000,0.000
STATE,0.000,0.000
HEALTHCARE_EXPENSES,0.000,0.000


This example keeps the same row count as the input table. For real projects, inspect utility and privacy risk before publishing synthetic data, especially when the source resembles patient-level records.